# Missingness and cohort retention


In [ ]:
from pathlib import Path
import os
SHARED_NOTEBOOK=os.environ.get("CHD_SHARED_NOTEBOOK",str(Path.cwd()/"00_shared_config.ipynb"))
if not Path(SHARED_NOTEBOOK).exists(): raise FileNotFoundError(SHARED_NOTEBOOK)
get_ipython().run_line_magic("run",f'"{SHARED_NOTEBOOK}"')


In [ ]:
from collections import OrderedDict
from datetime import datetime, timezone
import json
import os
import platform

from lifelines import CoxPHFitter, KaplanMeierFitter
from sklearn.impute import SimpleImputer
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sksurv.metrics import (
    brier_score,
    concordance_index_censored,
    concordance_index_ipcw,
)
from sksurv.util import Surv

AUDIT_NAME = "missingness_strategy_pilot_v1"
AUDIT_ROOT = Path(SENSITIVITY_OUTPUT_DIR) / "00_audit" / "missingness_strategy_pilot"
AUDIT_TABLE_DIR = AUDIT_ROOT / "tables"
AUDIT_FIGURE_DIR = AUDIT_ROOT / "figures"
for _path in [AUDIT_ROOT, AUDIT_TABLE_DIR, AUDIT_FIGURE_DIR]:
    _path.mkdir(parents=True, exist_ok=True)

AUDIT_SEED = int(os.environ.get("CHD_AUDIT_SEED", str(RANDOM_STATE)))
AUDIT_RUN_QUICK_COX = os.environ.get("CHD_AUDIT_RUN_QUICK_COX", "1") == "1"
AUDIT_MAX_TRAIN_N = int(os.environ.get("CHD_AUDIT_MAX_TRAIN_N", "20000"))
AUDIT_MAX_TEST_N = int(os.environ.get("CHD_AUDIT_MAX_TEST_N", "10000"))
AUDIT_COX_PENALIZER = float(os.environ.get("CHD_AUDIT_COX_PENALIZER", "1e-4"))

print("Audit:", AUDIT_NAME, flush=True)
print("Output:", AUDIT_ROOT, flush=True)
print("Quick Cox:", AUDIT_RUN_QUICK_COX, flush=True)
print("Pilot train/test caps:", AUDIT_MAX_TRAIN_N, AUDIT_MAX_TEST_N, flush=True)


In [ ]:
missing_input_paths=[os.path.join(DATA_DIR,name) for name in file_names if not os.path.exists(os.path.join(DATA_DIR,name))]
if missing_input_paths: raise FileNotFoundError("Required UKB input files missing:\n"+"\n".join(missing_input_paths))
input_file_manifest=[]
merge_audit_rows=[]

# 2. Decide columns and read CSV files

wanted_feature_ids = set()
for ids in feature_groups.values():
    wanted_feature_ids.update(ids)
if use_genetic_pcs:
    wanted_feature_ids.add(genetic_pc_id)

baseline_date_ids = [53]
icd10_code_ids = [41270]
icd10_date_ids = [41280]
icd9_code_ids = [41271]
icd9_date_ids = [41281]
death_cause_ids = [40001, 40002]
death_date_ids = [40000]
self_report_ids = [20002]
diabetes_ids = [2443]
medication_summary_ids = [6153, 6177]
medication_detail_ids = [20003]
genetic_qc_ids = reported_sex_ids + [22001, 22006] + qc_indicator_ids

wanted_exclusion_ids = set(
    baseline_date_ids
    + icd10_code_ids + icd10_date_ids
    + icd9_code_ids + icd9_date_ids
    + death_cause_ids + death_date_ids
    + self_report_ids
    + diabetes_ids
    + medication_summary_ids
    + medication_detail_ids
    + genetic_qc_ids
)
all_wanted_ids = wanted_feature_ids | wanted_exclusion_ids

def read_selected_cols(file_path):
    headers = pd.read_csv(file_path, nrows=0).columns.tolist()
    eid_col = find_eid_col(headers)
    if eid_col is None:
        raise ValueError(f"{file_path} has no Eid column")
    cols_to_use = [eid_col]
    for col in headers:
        if col == eid_col:
            continue
        parsed = parse_ukb_col(col)
        if parsed is None:
            continue
        fid, inst, arr = parsed
        if fid in death_cause_ids + death_date_ids:
            cols_to_use.append(col)
            continue
        if inst != 0:
            continue
        if fid in bp_ids:
            cols_to_use.append(col)
        elif use_genetic_pcs and fid == genetic_pc_id and arr in pc_numbers:
            cols_to_use.append(col)
        elif fid in (
            icd10_code_ids + icd10_date_ids
            + icd9_code_ids + icd9_date_ids
            + self_report_ids
            + medication_summary_ids
            + medication_detail_ids
        ):
            cols_to_use.append(col)
        elif fid in all_wanted_ids and arr == 0:
            cols_to_use.append(col)
    if len(cols_to_use) == 1:
        pass
        return None
    df = pd.read_csv(file_path, usecols=cols_to_use, low_memory=False)
    df = df.rename(columns={eid_col: "Eid"})
    pass
    return df

dataframes = []
for file_name in file_names:
    path = os.path.join(DATA_DIR, file_name)
    if not os.path.exists(path):
        raise FileNotFoundError(f"Missing input file: {path}")
        continue
    df_part = read_selected_cols(path)
    if df_part is not None:
        dataframes.append(df_part)
        input_file_manifest.append({"name":file_name,"path":path,"n":int(len(df_part)),"columns":int(df_part.shape[1]),"sha256":sha256_file(path)})

if len(dataframes) == 0:
    raise FileNotFoundError("No usable inputs. Check CHD_DATA_DIR and the input file names.")

df_final = dataframes[0]
for df_part in dataframes[1:]:
    overlap_cols = sorted((set(df_final.columns) & set(df_part.columns)) - {"Eid"})
    if overlap_cols:
        overlap_check=pd.merge(df_final[["Eid"]+overlap_cols],df_part[["Eid"]+overlap_cols],on="Eid",how="inner",suffixes=("__left","__right"))
        for duplicate in overlap_cols:
            left=overlap_check[duplicate+"__left"]; right=overlap_check[duplicate+"__right"]
            comparable=left.notna() & right.notna(); left_number=pd.to_numeric(left,errors="coerce"); right_number=pd.to_numeric(right,errors="coerce")
            numeric=comparable & left_number.notna() & right_number.notna(); nonnumeric=comparable & ~numeric
            numeric_mismatch=~np.isclose(left_number.loc[numeric],right_number.loc[numeric],rtol=1e-10,atol=1e-12,equal_nan=True)
            string_mismatch=left.loc[nonnumeric].astype(str).str.strip()!=right.loc[nonnumeric].astype(str).str.strip()
            mismatches=int(numeric_mismatch.sum()+string_mismatch.sum())
            merge_audit_rows.append({"column":duplicate,"overlap_n":int(comparable.sum()),"mismatch_n":mismatches})
            if mismatches: raise RuntimeError(f"Conflicting duplicate field {duplicate}: {mismatches} values")
        pass
        df_part = df_part.drop(columns=overlap_cols)
    df_final = pd.merge(df_final, df_part, on="Eid", how="inner")

print("-" * 40)

atomic_write_csv(pd.DataFrame(input_file_manifest),os.path.join(AUDIT_DIR,"input_file_manifest.csv"),index=False)
atomic_write_csv(pd.DataFrame(merge_audit_rows),os.path.join(AUDIT_DIR,"duplicate_field_audit.csv"),index=False)


In [ ]:
# 3. Clean feature columns and build feature_cols

for bp_id in bp_ids:
    pattern = rf"^{bp_id}-0\.\d+$"
    bp_cols = [c for c in df_final.columns if re.match(pattern, str(c).strip().replace('"', ""))]
    if len(bp_cols) == 0:
        pass
        continue
    pass
    for c in bp_cols:
        df_final[c] = pd.to_numeric(df_final[c], errors="coerce")
    df_final[str(bp_id)] = df_final[bp_cols].mean(axis=1)
    df_final = df_final.drop(columns=bp_cols)

rename_dict = {}
for col in df_final.columns:
    if col == "Eid":
        continue
    if col in [str(x) for x in bp_ids]:
        continue
    parsed = parse_ukb_col(col)
    if parsed is None:
        continue
    fid, inst, arr = parsed
    if fid == genetic_pc_id and inst == 0 and arr in pc_numbers:
        rename_dict[col] = f"{genetic_pc_id}_PC{arr}"
    elif fid in wanted_feature_ids and inst == 0 and arr == 0:
        rename_dict[col] = str(fid)
df_final = df_final.rename(columns=rename_dict)

feature_cols = []
for _, ids in feature_groups.items():
    for fid in ids:
        feature_cols.append(str(fid))
if use_genetic_pcs:
    for pc in pc_numbers:
        feature_cols.append(f"{genetic_pc_id}_PC{pc}")
feature_cols = list(dict.fromkeys(feature_cols))

missing_feature_cols = [c for c in feature_cols if c not in df_final.columns]
if len(missing_feature_cols) > 0:
    print("-" * 40)
    pass
    print(missing_feature_cols)
    raise ValueError("Required predictors are missing. Check input files and field IDs.")

for c in feature_cols:
    df_final[c] = pd.to_numeric(df_final[c], errors="coerce")

special_missing_values = {"20116": [-3, -1]}
for col, bad_values in special_missing_values.items():
    if col in df_final.columns:
        df_final[col] = df_final[col].replace(bad_values, np.nan)

nmr106_cols = [str(x) for x in NMR_106_FIELD_MAP.values()]
nmr106_feature_cols = set(nmr106_cols)
non_nmr_feature_cols = [c for c in feature_cols if c not in nmr106_feature_cols]

print("-" * 40)


In [ ]:
# 4. Build and apply exclusion masks

exclude_reasons = {}
baseline_date_cols = get_cols_by_fid(df_final, 53, instance=0)
if len(baseline_date_cols) == 0:
    raise ValueError("Baseline assessment date field 53-0.0 is required to date ICD diagnoses.")
baseline_date_col = baseline_date_cols[0]
baseline_date = pd.to_datetime(df_final[baseline_date_col], errors="coerce")

self_report_cols = get_cols_by_fid(df_final, 20002, instance=0)
mask_self_report_chd = any_value_in_cols(df_final, self_report_cols, baseline_chd_self_report_codes)
exclude_reasons["baseline_chd_self_report_20002"] = mask_self_report_chd

has_icd10_event, event_date_icd10 = get_first_event_date_from_icd(df_final, 41270, 41280, target_icd10_prefixes)
has_icd9_event, event_date_icd9 = get_first_event_date_from_icd(df_final, 41271, 41281, target_icd9_prefixes)
event_date_union = pd.concat([event_date_icd10, event_date_icd9], axis=1).min(axis=1)

history_from_self = mask_self_report_chd.copy()
history_icd10_only = history_from_self | (
    event_date_icd10.notna() & baseline_date.notna() & (event_date_icd10 <= baseline_date)
)
future_event_icd10_only = event_date_icd10.notna() & baseline_date.notna() & (event_date_icd10 > baseline_date)
future_event_icd10_only.loc[history_icd10_only] = False
history_union = history_from_self | (
    event_date_union.notna() & baseline_date.notna() & (event_date_union <= baseline_date)
)
future_event_union = event_date_union.notna() & baseline_date.notna() & (event_date_union > baseline_date)
future_event_union.loc[history_union] = False
icd9_only = has_icd9_event & (~has_icd10_event)
icd9_earlier_than_icd10 = event_date_icd9.notna() & event_date_icd10.notna() & (event_date_icd9 < event_date_icd10)
reclassified_by_icd9 = (history_union != history_icd10_only) | (future_event_union != future_event_icd10_only)

print("-" * 40)

exclude_reasons["baseline_chd_icd10_I20_I25"] = (
    event_date_icd10.notna() & baseline_date.notna() & (event_date_icd10 <= baseline_date)
)
exclude_reasons["baseline_chd_icd9_410_414"] = (
    event_date_icd9.notna() & baseline_date.notna() & (event_date_icd9 <= baseline_date)
)

med_summary_cols = get_cols_by_fid(df_final, 6153, instance=0) + get_cols_by_fid(df_final, 6177, instance=0)
exclude_reasons["baseline_lipid_lowering_6153_6177"] = any_value_in_cols(
    df_final, med_summary_cols, lipid_lowering_values_6153_6177
)
med_detail_cols = get_cols_by_fid(df_final, 20003, instance=0)
exclude_reasons["baseline_lipid_lowering_20003"] = any_value_in_cols(
    df_final, med_detail_cols, lipid_lowering_codes_20003
)
diabetes_cols = get_cols_by_fid(df_final, 2443, instance=0)
exclude_reasons["baseline_diabetes_2443"] = any_value_in_cols(
    df_final, diabetes_cols, diabetes_yes_values_2443
)
diabetes_self_report_codes_main = (
    baseline_diabetes_self_report_codes_conservative
    if EXCLUDE_BASELINE_GESTATIONAL_DIABETES_IN_MAIN
    else baseline_diabetes_self_report_codes
)
exclude_reasons["baseline_diabetes_self_report_20002"] = any_value_in_cols(
    df_final, self_report_cols, diabetes_self_report_codes_main
)

if not EXCLUDE_BASELINE_DIABETES:
    exclude_reasons["baseline_diabetes_2443"] = pd.Series(False, index=df_final.index)
    exclude_reasons["baseline_diabetes_self_report_20002"] = pd.Series(False, index=df_final.index)
if not EXCLUDE_BASELINE_LIPID_LOWERING:
    exclude_reasons["baseline_lipid_lowering_6153_6177"] = pd.Series(False, index=df_final.index)
    exclude_reasons["baseline_lipid_lowering_20003"] = pd.Series(False, index=df_final.index)

if APPLY_GENETIC_QC:
    genetic_sex_cols = get_cols_by_fid(df_final, 22001, instance=0)
    if "31" in df_final.columns:
        sex_reported = pd.to_numeric(df_final["31"], errors="coerce")
    else:
        reported_sex_cols_found = get_cols_by_fid(df_final, 31, instance=0)
        sex_reported = pd.to_numeric(df_final[reported_sex_cols_found[0]], errors="coerce") if len(reported_sex_cols_found) > 0 else None
    if sex_reported is not None and len(genetic_sex_cols) > 0:
        sex_genetic = pd.to_numeric(df_final[genetic_sex_cols[0]], errors="coerce")
        mask_sex_mismatch = sex_reported.notna() & sex_genetic.notna() & (sex_reported != sex_genetic)
    else:
        mask_sex_mismatch = pd.Series(False, index=df_final.index)
    exclude_reasons["qc_22001_sex_mismatch"] = mask_sex_mismatch

    ethnic_cols = get_cols_by_fid(df_final, 22006, instance=0)
    if ALLOWED_22006_VALUES is not None and len(ethnic_cols) > 0:
        ethnic_s = pd.to_numeric(df_final[ethnic_cols[0]], errors="coerce")
        mask_ethnic = ethnic_s.notna() & ~ethnic_s.isin(ALLOWED_22006_VALUES)
    else:
        mask_ethnic = pd.Series(False, index=df_final.index)
    exclude_reasons["qc_22006_genetic_ethnic_grouping"] = mask_ethnic

    for fid in qc_indicator_ids:
        exclude_reasons[f"qc_{fid}_indicator"] = indicator_abnormal_mask(df_final, fid)

# Mutually exclusive sequential counts plus overlap audit.
_reason_names=list(exclude_reasons)
_overlap=np.zeros((len(_reason_names),len(_reason_names)),dtype=int)
for i,left_name in enumerate(_reason_names):
    for j,right_name in enumerate(_reason_names): _overlap[i,j]=int((exclude_reasons[left_name].fillna(False)&exclude_reasons[right_name].fillna(False)).sum())
atomic_write_csv(pd.DataFrame(_overlap,index=_reason_names,columns=_reason_names).reset_index(names="Reason"),os.path.join(AUDIT_DIR,"exclusion_overlap_matrix.csv"),index=False)
_remaining=pd.Series(True,index=df_final.index); sequential_exclusion_rows=[{"Step":"Initial merged records","N":int(len(df_final)),"Excluded_at_step":0}]
for _reason in _reason_names:
    _new=_remaining & exclude_reasons[_reason].fillna(False); _remaining &= ~exclude_reasons[_reason].fillna(False)
    sequential_exclusion_rows.append({"Step":"Exclude: "+_reason,"N":int(_remaining.sum()),"Excluded_at_step":int(_new.sum())})
atomic_write_csv(pd.DataFrame(sequential_exclusion_rows),os.path.join(AUDIT_DIR,"sequential_exclusion_counts.csv"),index=False)

exclude_mask = pd.Series(False, index=df_final.index)
print("-" * 40)

for reason, mask in exclude_reasons.items():
    mask = mask.fillna(False)
    exclude_mask = exclude_mask | mask
    print(f"{reason}: {int(mask.sum())}")

before_exclusion_n = len(df_final)
df_final = df_final.loc[~exclude_mask].copy()
after_exclusion_n = len(df_final)
print("-" * 40)


In [ ]:
# 5. Build full-follow-up CHD Event / Time

def get_all_cause_death_date(df):
    """Use the earliest death date across instances of UK Biobank field 40000."""
    death_date_cols_found = get_cols_by_fid_any_instance(df, 40000)
    if len(death_date_cols_found) == 0:
        return pd.Series(pd.NaT, index=df.index, dtype="datetime64[ns]")
    death_dates = [
        pd.to_datetime(df[column], errors="coerce")
        for column in death_date_cols_found
    ]
    return rowwise_min_datetime(death_dates, index=df.index)

def get_chd_death_date(df, all_cause_death_date):
    """Identify CAD-related death dates using the cause-of-death fields."""
    cause_columns = (
        get_cols_by_fid_any_instance(df, 40001)
        + get_cols_by_fid_any_instance(df, 40002)
    )
    result = pd.Series(pd.NaT, index=df.index, dtype="datetime64[ns]")
    for cause_column in cause_columns:
        parsed = parse_ukb_col(cause_column)
        paired_date_column = f"40000-{parsed[1]}.0" if parsed is not None else None
        if paired_date_column in df.columns:
            this_date = pd.to_datetime(df[paired_date_column], errors="coerce")
        else:
            this_date = all_cause_death_date

        cause = df[cause_column].astype("string").str.upper().str.strip()
        is_chd = cause.str.startswith(tuple(target_icd10_prefixes), na=False)
        update = is_chd & this_date.notna() & (
            result.isna() | (this_date < result)
        )
        result.loc[update] = this_date.loc[update]
    return result

registry_end_date = pd.to_datetime(REGISTRY_END_DATE_STR)
df_outcome = df_final.copy()
df_outcome["Date_Attending"] = baseline_date.loc[df_outcome.index]

all_cause_death_date = get_all_cause_death_date(df_outcome)
chd_death_date = get_chd_death_date(df_outcome, all_cause_death_date)
first_chd_date = rowwise_min_datetime(
    [event_date_union.loc[df_outcome.index], chd_death_date],
    index=df_outcome.index,
)

registry_end = pd.Series(
    registry_end_date,
    index=df_outcome.index,
    dtype="datetime64[ns]",
)
censor_date = rowwise_min_datetime(
    [registry_end, all_cause_death_date],
    index=df_outcome.index,
)

df_outcome["First_CHD_Date"] = first_chd_date
df_outcome["All_Cause_Death_Date"] = all_cause_death_date
df_outcome["CHD_Death_Date"] = chd_death_date
df_outcome["Censor_Date"] = censor_date

event_mask = (
    df_outcome["First_CHD_Date"].notna()
    & df_outcome["Date_Attending"].notna()
    & df_outcome["Censor_Date"].notna()
    & (df_outcome["First_CHD_Date"] > df_outcome["Date_Attending"])
    & (df_outcome["First_CHD_Date"] <= df_outcome["Censor_Date"])
)

df_outcome["Event"] = event_mask.astype(int)
df_outcome["Outcome_Date"] = df_outcome["Censor_Date"]
df_outcome.loc[event_mask, "Outcome_Date"] = df_outcome.loc[
    event_mask, "First_CHD_Date"
]
df_outcome["Time"] = (
    df_outcome["Outcome_Date"] - df_outcome["Date_Attending"]
).dt.days / 365.25

baseline_chd_leak_mask = (
    df_outcome["First_CHD_Date"].notna()
    & df_outcome["Date_Attending"].notna()
    & (df_outcome["First_CHD_Date"] <= df_outcome["Date_Attending"])
)
invalid_time_mask = df_outcome["Time"].isna() | (df_outcome["Time"] <= 0)

before_filter_n = len(df_outcome)
df_outcome = df_outcome.loc[
    ~baseline_chd_leak_mask & ~invalid_time_mask
].copy()

print("-" * 40)

print(df_outcome["Event"].value_counts().sort_index())
print("Minimum, median and maximum follow-up:",
      df_outcome["Time"].min(),
      df_outcome["Time"].median(),
      df_outcome["Time"].max())

_non_chd_death=(df_outcome["Event"].eq(0)&df_outcome["All_Cause_Death_Date"].notna()&(df_outcome["All_Cause_Death_Date"]>df_outcome["Date_Attending"])&(df_outcome["All_Cause_Death_Date"]<=pd.to_datetime(REGISTRY_END_DATE_STR)))
df_outcome["Competing_Event"]=_non_chd_death.astype(int)
df_outcome["Event_Type"]=np.select([df_outcome.Event.eq(1),df_outcome.Competing_Event.eq(1)],["CHD","Non-CHD death"],default="Censored")


In [ ]:
# Build prespecified clinical variables; do not add data-driven interactions to the benchmark.
smoking=pd.to_numeric(df_outcome["20116"],errors="coerce")
for name,level in [("smoking_previous",1),("smoking_current",2)]:
    values=pd.Series(np.nan,index=df_outcome.index,dtype=float)
    values.loc[smoking.notna()]=(smoking.loc[smoking.notna()]==level).astype(float)
    df_outcome[name]=values
for column in ["smoking_previous","smoking_current"]:
    if column not in CATEGORICAL_FEATURE_COLS: CATEGORICAL_FEATURE_COLS.append(column)

clinical_base=["31","21003","21001","4079","4080","30690","30760","30700","30720","smoking_previous","smoking_current"]
pc_cols=[f"{genetic_pc_id}_PC{index}" for index in pc_numbers]
prs_col=PRS_COL
missing_prespecified=[column for column in clinical_base+pc_cols+[prs_col]+nmr106_cols if column not in df_outcome.columns]
if missing_prespecified: raise KeyError(f"Required analysis variables missing: {missing_prespecified}")


In [ ]:
# Construct the candidate population and all prespecified cohort masks.
df_candidate = df_outcome.copy().reset_index(drop=True)
df_candidate["NMR_Nonmissing_N"] = df_candidate[nmr106_cols].notna().sum(axis=1)
df_candidate["NMR_Nonmissing_Rate"] = (
    df_candidate["NMR_Nonmissing_N"] / len(nmr106_cols)
)
df_candidate["PRS_Present"] = pd.to_numeric(
    df_candidate[prs_col], errors="coerce"
).notna()

# Exact historical denominator: pre-derived clinical variables + PCs + PRS + Raw106.
# This should contain 127 variables for the current specification.
legacy_clinical_source = [
    "31", "21003", "21001", "4079", "4080", "30690", "30760",
    "30700", "30720", "20116",
]
legacy_missingness_cols = list(dict.fromkeys(
    legacy_clinical_source + pc_cols + [prs_col] + nmr106_cols
))
missing_legacy_cols = [c for c in legacy_missingness_cols if c not in df_candidate.columns]
if missing_legacy_cols:
    raise KeyError("Historical missingness denominator columns absent: " + ", ".join(missing_legacy_cols))

df_candidate["Legacy_Row_Missing_N"] = (
    df_candidate[legacy_missingness_cols].isna().sum(axis=1)
)
df_candidate["Legacy_Row_Missing_Rate"] = (
    df_candidate["Legacy_Row_Missing_N"] / len(legacy_missingness_cols)
)

strategy_masks = OrderedDict([
    (
        "Legacy_row_missing_le10_exact",
        df_candidate["Legacy_Row_Missing_Rate"].le(0.10),
    ),
    (
        "Legacy_row_missing_le10_plus_PRS",
        df_candidate["Legacy_Row_Missing_Rate"].le(0.10)
        & df_candidate["PRS_Present"],
    ),
    (
        "Raw106_complete_plus_PRS",
        df_candidate["NMR_Nonmissing_Rate"].eq(1.0)
        & df_candidate["PRS_Present"],
    ),
    (
        "Raw106_coverage95_plus_PRS",
        df_candidate["NMR_Nonmissing_Rate"].ge(0.95)
        & df_candidate["PRS_Present"],
    ),
    (
        "Raw106_coverage90_plus_PRS",
        df_candidate["NMR_Nonmissing_Rate"].ge(0.90)
        & df_candidate["PRS_Present"],
    ),
    (
        "All127_complete_plus_PRS",
        df_candidate["Legacy_Row_Missing_Rate"].eq(0.0)
        & df_candidate["PRS_Present"],
    ),
])

if len(nmr106_cols) != 106:
    raise RuntimeError(f"Expected exactly 106 Raw NMR fields, found {len(nmr106_cols)}")
if len(legacy_missingness_cols) != 127:
    raise RuntimeError(
        f"Expected the historical denominator to contain 127 variables, found {len(legacy_missingness_cols)}"
    )

print("Candidate N:", len(df_candidate), "Events:", int(df_candidate["Event"].sum()), flush=True)
print("Raw106 fields:", len(nmr106_cols), flush=True)
print("Historical denominator fields:", len(legacy_missingness_cols), flush=True)


In [ ]:
# Sample retention, event retention, and missingness summaries.
summary_rows = []
candidate_n = len(df_candidate)
candidate_events = int(df_candidate["Event"].sum())
for strategy, mask in strategy_masks.items():
    included = df_candidate.loc[mask]
    event_n = int(included["Event"].sum())
    summary_rows.append({
        "Strategy": strategy,
        "Candidate_N": int(candidate_n),
        "Included_N": int(len(included)),
        "Excluded_N": int(candidate_n - len(included)),
        "Retained_Percent": 100.0 * len(included) / candidate_n,
        "Candidate_Event_N": candidate_events,
        "Included_Event_N": event_n,
        "Event_Retention_Percent": (
            100.0 * event_n / candidate_events if candidate_events else np.nan
        ),
        "Included_Event_Rate": float(included["Event"].mean()) if len(included) else np.nan,
        "PRS_Missing_Included_N": int((~included["PRS_Present"]).sum()),
        "Mean_NMR106_Coverage": float(included["NMR_Nonmissing_Rate"].mean()) if len(included) else np.nan,
        "Median_NMR106_Coverage": float(included["NMR_Nonmissing_Rate"].median()) if len(included) else np.nan,
        "Mean_Legacy_Row_Missing_Rate": float(included["Legacy_Row_Missing_Rate"].mean()) if len(included) else np.nan,
    })

strategy_summary = pd.DataFrame(summary_rows)
atomic_write_csv(
    strategy_summary,
    AUDIT_TABLE_DIR / "strategy_sample_event_retention.csv",
    index=False,
)
display(strategy_summary)

coverage_quantiles = (
    df_candidate["NMR_Nonmissing_Rate"]
    .quantile([0, 0.01, 0.05, 0.10, 0.25, 0.50, 0.75, 0.90, 0.95, 0.99, 1.0])
    .rename_axis("Quantile")
    .reset_index(name="NMR106_Nonmissing_Rate")
)
atomic_write_csv(
    coverage_quantiles,
    AUDIT_TABLE_DIR / "nmr106_coverage_quantiles.csv",
    index=False,
)

nmr_feature_missingness = pd.DataFrame({
    "Field": nmr106_cols,
    "Missing_N": df_candidate[nmr106_cols].isna().sum().to_numpy(),
    "Missing_Rate": df_candidate[nmr106_cols].isna().mean().to_numpy(),
}).sort_values(["Missing_Rate", "Field"], ascending=[False, True])
atomic_write_csv(
    nmr_feature_missingness,
    AUDIT_TABLE_DIR / "nmr106_feature_missingness.csv",
    index=False,
)
display(coverage_quantiles)
display(nmr_feature_missingness.head(20))
